In [63]:
# Preparing pathing
%load_ext autoreload
%autoreload 2
from titanic_ml import paths
import matplotlib.pyplot as plt
import pandas as pd
from titanic_ml.common.data.eda import summarize_categorical_column, summarize_numerical_column
from titanic_ml.common.data.eda import run_eda 
TARGET = "Survived"

from ast import For
from cmath import exp

from titanic_ml.common.experiments.runner import run_experiments, run_experiment_group_workflow
from titanic_ml.common.experiments.config import ALL_EXPERIMENTS
from titanic_ml.common.experiments.report import experiment_report, experiment_group_summary_report, baseline_summary_to_markdown, workflow_report
from titanic_ml.common.experiments.save_load import save_results, load_results, save_configs, load_configs
from titanic_ml.common.experiments.compare import leaderboard, compare_experiment_groups, summarize_group_comparison, titanic_notes_leaderboard


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [64]:


df = pd.read_csv(paths.TRAIN_PATH)


# Line to rerun all experiments to update the results with the latest code changes. This will take a while.
# Uncomment to run all experiments and update results.
# for Name, exp_config in config.ALL_EXPERIMENTS:
#     print(f"Running {Name} experiments...")
#     exp_result = run_experiments(df, exp_config, target=TARGET, verbose=True, debug=True)
#     save_results(exp_result)
#     save_configs(exp_config)



In [65]:
exp_config = ALL_EXPERIMENTS["baseline__raw"]
exp_result = run_experiments(df, exp_config, target=TARGET, verbose=True, debug=True)
exp_report = experiment_report(exp_result, exp_config, print_report=True)
save_results(exp_result)
save_configs(exp_config)

Running experiment: baseline__raw__logreg

Experiment 'baseline__raw__logreg' results:
  stage: baseline
  feature_group: raw
  model_name: logreg
  group: baseline__raw
  status: success
  error_type: 
  error_message: 
  test_accuracy_mean: 0.786
  test_accuracy_std: 0.018
  train_accuracy_mean: 0.803
  train_accuracy_std: 0.005
  test_precision_mean: 0.736
  test_precision_std: 0.036
  train_precision_mean: 0.762
  train_precision_std: 0.011
  test_recall_mean: 0.693
  test_recall_std: 0.038
  train_recall_mean: 0.708
  train_recall_std: 0.015
  test_f1_mean: 0.713
  test_f1_std: 0.026
  train_f1_mean: 0.734
  train_f1_std: 0.008
  fit_time_mean: 0.034
  score_time_mean: 0.031
  notes: Base logistic regression, using raw configuration. Baseline for comparison.

Experiment 'baseline__raw__logreg' completed.
----------------------------------------
Running experiment: baseline__raw__knn

Experiment 'baseline__raw__knn' results:
  stage: baseline
  feature_group: raw
  model_name: knn


{'fe01__family__logreg': {'name': 'fe01__family__logreg',
  'features': ['Pclass',
   'Sex',
   'Age',
   'Fare',
   'Embarked',
   'FamilySize',
   'IsAlone'],
  'feature_engineering': ['add_family_features'],
  'preprocessing': {'numeric_features': ['Age',
    'Fare',
    'FamilySize',
    'IsAlone'],
   'onehot_features': ['Sex', 'Embarked'],
   'ordinal_features': ['Pclass'],
   'numeric_imputer': 'median',
   'categorical_imputer': 'most_frequent',
   'scaler': 'standard'},
  'model_name': 'logreg',
  'model_params': {'max_iter': 1000, 'random_state': 42},
  'evaluation': {'method': 'cross_validation',
   'cv': 5,
   'scoring': ['accuracy', 'precision', 'recall', 'f1'],
   'return_train_score': True,
   'n_jobs': -1},
  'notes': 'Feature engineering 01: replaces SibSp/Parch with FamilySize and IsAlone.',
  'stage': 'fe01',
  'feature_group': 'family',
  'group': 'fe01__family'},
 'fe01__family__knn': {'name': 'fe01__family__knn',
  'features': ['Pclass',
   'Sex',
   'Age',
   'Fa

In [66]:
baseline_summary = baseline_summary_to_markdown(exp_result)
print("Baseline summary:")
print(baseline_summary)

Baseline summary:
| model_name    | accuracy      | f1            |
|:--------------|:--------------|:--------------|
| logreg        | 0.786 ± 0.018 | 0.713 ± 0.026 |
| knn           | 0.809 ± 0.021 | 0.742 ± 0.026 |
| svc           | 0.827 ± 0.015 | 0.76 ± 0.026  |
| decision_tree | 0.803 ± 0.023 | 0.702 ± 0.055 |
| random_forest | 0.822 ± 0.02  | 0.744 ± 0.041 |
| extra_trees   | 0.804 ± 0.012 | 0.721 ± 0.025 |
| xgb           | 0.826 ± 0.025 | 0.758 ± 0.041 |


In [67]:

all_results = load_results()
print("Loaded results:")
print(all_results)

all_configs = load_configs()
print("Loaded configs:")
print(all_configs)

Loaded results:
                      experiment     stage feature_group     model_name  \
0           fe01__family__logreg      fe01        family         logreg   
1              fe01__family__knn      fe01        family            knn   
2              fe01__family__svc      fe01        family            svc   
3    fe01__family__decision_tree      fe01        family  decision_tree   
4    fe01__family__random_forest      fe01        family  random_forest   
5      fe01__family__extra_trees      fe01        family    extra_trees   
6              fe01__family__xgb      fe01        family            xgb   
7          baseline__raw__logreg  baseline           raw         logreg   
8             baseline__raw__knn  baseline           raw            knn   
9             baseline__raw__svc  baseline           raw            svc   
10  baseline__raw__decision_tree  baseline           raw  decision_tree   
11  baseline__raw__random_forest  baseline           raw  random_forest   
12    bas

In [68]:
exp_config = ALL_EXPERIMENTS["fe01__family"]
exp_result = run_experiments(df, exp_config, target=TARGET, verbose=True, debug=True)
exp_report = experiment_report(exp_result, exp_config, print_report=True)
save_results(exp_result)
save_configs(exp_config)

reference_groups = "baseline__raw"
compare_groups = ["fe01__family"]
top_n = 10
top_metric = "test_accuracy_mean"
metrics = ["test_accuracy_mean", "test_f1_mean"]

all_results = load_results()

comparison = compare_experiment_groups(
    all_results,
    reference_group=reference_groups,
    compare_groups=compare_groups,
    metrics=metrics,
)

summary = summarize_group_comparison(comparison, metrics=metrics)

top_models = leaderboard(all_results, metric=top_metric, top_n=top_n)

print(f"Comparison between {reference_groups} and {compare_groups}:")
print(comparison)
print()
print("Summary of comparison:")
print(summary)
print()
print(f"Top {top_n} models:")
print(top_models)


Running experiment: fe01__family__logreg
Applying feature: add_family_features
Features after add_family_features: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'FamilySize', 'IsAlone']

Experiment 'fe01__family__logreg' results:
  stage: fe01
  feature_group: family
  model_name: logreg
  group: fe01__family
  status: success
  error_type: 
  error_message: 
  test_accuracy_mean: 0.795
  test_accuracy_std: 0.016
  train_accuracy_mean: 0.806
  train_accuracy_std: 0.005
  test_precision_mean: 0.755
  test_precision_std: 0.038
  train_precision_mean: 0.766
  train_precision_std: 0.007
  test_recall_mean: 0.693
  test_recall_std: 0.038
  train_recall_mean: 0.711
  train_recall_std: 0.012
  test_f1_mean: 0.721
  test_f1_std: 0.021
  train_f1_mean: 0.738
  train_f1_std: 0.008
  fit_time_mean: 0.037
  score_time_mean: 0.038
  notes: Feature engineering 01: replaces SibSp/Parch with FamilySize and IsAlone.

Experiment 'fe0

In [69]:
Leaderboard = leaderboard(all_results, metric="test_accuracy_mean", top_n=5)
print("Leaderboard:")
print(leaderboard)

Leaderboard:
<function leaderboard at 0x00000275CED16440>


In [70]:
comparison = comparison = compare_experiment_groups(
    results_df=all_results,
    reference_group="baseline__raw",
    compare_groups=["fe01__family"],
)
print("Comparison of feature engineering groups:")
print(comparison)

Comparison of feature engineering groups:
  reference_group compare_group     model_name  test_accuracy_mean_reference  \
0   baseline__raw  fe01__family         logreg                         0.786   
1   baseline__raw  fe01__family            knn                         0.809   
2   baseline__raw  fe01__family            svc                         0.827   
3   baseline__raw  fe01__family  decision_tree                         0.803   
4   baseline__raw  fe01__family  random_forest                         0.822   
5   baseline__raw  fe01__family    extra_trees                         0.804   
6   baseline__raw  fe01__family            xgb                         0.826   

   test_accuracy_mean_compare  test_accuracy_mean_delta  \
0                       0.795                     0.009   
1                       0.805                    -0.004   
2                       0.826                    -0.001   
3                       0.806                     0.003   
4                     

In [71]:
summary = summarize_group_comparison(comparison)
print("Summary of comparison:")
print(summary)

Summary of comparison:
  compare_group  test_accuracy_mean_delta_mean  test_accuracy_mean_delta_min  \
0  fe01__family                       0.000429                        -0.006   

   test_accuracy_mean_delta_max  test_precision_mean_delta_mean  \
0                         0.009                        0.000857   

   test_precision_mean_delta_min  test_precision_mean_delta_max  \
0                         -0.008                          0.019   

   test_recall_mean_delta_mean  test_recall_mean_delta_min  \
0                     0.000714                      -0.009   

   test_recall_mean_delta_max  test_f1_mean_delta_mean  \
0                       0.023                 0.000429   

   test_f1_mean_delta_min  test_f1_mean_delta_max  
0                  -0.009                    0.01  


In [72]:
reference_groups = "baseline__raw"
compare_groups = ["fe01__family"]
top_n = 10
top_metric = "test_accuracy_mean"
metrics = ["test_accuracy_mean", "test_f1_mean"]

all_results = load_results()

comparison = compare_experiment_groups(
    all_results,
    reference_group=reference_groups,
    compare_groups=compare_groups,
    metrics=metrics,
)

summary = summarize_group_comparison(comparison, metrics=metrics)

top_models = leaderboard(all_results, metric=top_metric, top_n=top_n)

print(f"Comparison between {reference_groups} and {compare_groups}:")
print(comparison)
print()
print("Summary of comparison:")
print(summary)
print()
print(f"Top {top_n} models:")
print(top_models)



Comparison between baseline__raw and ['fe01__family']:
  reference_group compare_group     model_name  test_accuracy_mean_reference  \
0   baseline__raw  fe01__family         logreg                         0.786   
1   baseline__raw  fe01__family            knn                         0.809   
2   baseline__raw  fe01__family            svc                         0.827   
3   baseline__raw  fe01__family  decision_tree                         0.803   
4   baseline__raw  fe01__family  random_forest                         0.822   
5   baseline__raw  fe01__family    extra_trees                         0.804   
6   baseline__raw  fe01__family            xgb                         0.826   

   test_accuracy_mean_compare  test_accuracy_mean_delta  \
0                       0.795                     0.009   
1                       0.805                    -0.004   
2                       0.826                    -0.001   
3                       0.806                     0.003   
4        

In [73]:
report = experiment_group_summary_report(
    results_df=all_results,
    reference_group="baseline__raw",
    compare_group="fe01__family",
    description="Tests whether FamilySize and IsAlone replace SibSp/Parch effectively.",
    conclusion="Small/negligible impact overall. LogReg improved slightly.",
)

In [74]:
print(report)

### fe01__family

Tests whether FamilySize and IsAlone replace SibSp/Parch effectively.

<details>
<summary>Comparison details</summary>

#### Comparison vs baseline

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe01__family    | logreg        |                          0.786 |                        0.795 |                      0.009 |                    0.713 |                  0.721 |                0.008 |
| baseline__raw     | fe01__family    | knn           |                          0.809 |                        0.805 |                     -0.004 |             

In [75]:
print(f"Top {top_n} models:")
print(top_models)

Top 10 models:
                     experiment          group     model_name  \
0            baseline__raw__svc  baseline__raw            svc   
1            baseline__raw__xgb  baseline__raw            xgb   
2             fe01__family__svc   fe01__family            svc   
3             fe01__family__xgb   fe01__family            xgb   
4  baseline__raw__random_forest  baseline__raw  random_forest   
5   fe01__family__random_forest   fe01__family  random_forest   
6            baseline__raw__knn  baseline__raw            knn   
7   fe01__family__decision_tree   fe01__family  decision_tree   
8     fe01__family__extra_trees   fe01__family    extra_trees   
9             fe01__family__knn   fe01__family            knn   

   test_accuracy_mean                                              notes  
0               0.827  Base SVC, using raw configuration. Baseline fo...  
1               0.826  Base xgb, using raw configuration. Baseline fo...  
2               0.826  Feature engineering 0

In [76]:
titanic_notes_top_models = titanic_notes_leaderboard(all_results, top_n=10)
print("Top models on Titanic Notes leaderboard:")
print()
print(titanic_notes_top_models)

Top models on Titanic Notes leaderboard:

| experiment                   | model_name    |   test_accuracy_mean |   test_f1_mean |
|:-----------------------------|:--------------|---------------------:|---------------:|
| baseline__raw__svc           | svc           |                0.827 |          0.76  |
| baseline__raw__xgb           | xgb           |                0.826 |          0.758 |
| fe01__family__svc            | svc           |                0.826 |          0.756 |
| fe01__family__xgb            | xgb           |                0.826 |          0.756 |
| baseline__raw__random_forest | random_forest |                0.822 |          0.744 |
| fe01__family__random_forest  | random_forest |                0.816 |          0.735 |
| baseline__raw__knn           | knn           |                0.809 |          0.742 |
| fe01__family__decision_tree  | decision_tree |                0.806 |          0.712 |
| fe01__family__extra_trees    | extra_trees   |                0.80

In [77]:
workflow = run_experiment_group_workflow(
    df=df,
    experiment_configs=ALL_EXPERIMENTS["fe01__family"],
    target="Survived",
)

print("Workflow completed. Here are the results:")
print("Comparison between baseline and feature engineering group:")
print(workflow["comparison"])
print("Summary of comparison:")
print(workflow["summary"])
print("Leaderboard:")
print(workflow["leaderboard"])

Workflow completed. Here are the results:
Comparison between baseline and feature engineering group:
  reference_group compare_group     model_name  test_accuracy_mean_reference  \
0   baseline__raw  fe01__family         logreg                         0.786   
1   baseline__raw  fe01__family            knn                         0.809   
2   baseline__raw  fe01__family            svc                         0.827   
3   baseline__raw  fe01__family  decision_tree                         0.803   
4   baseline__raw  fe01__family  random_forest                         0.822   
5   baseline__raw  fe01__family    extra_trees                         0.804   
6   baseline__raw  fe01__family            xgb                         0.826   

   test_accuracy_mean_compare  test_accuracy_mean_delta  \
0                       0.795                     0.009   
1                       0.805                    -0.004   
2                       0.826                    -0.001   
3                     

In [78]:
full_report = workflow_report(workflow)

print("Full workflow report:")
print()
print('Report')
print(full_report['report'])
print()
print('Leaderboard')
print(full_report['leaderboard'])


Full workflow report:

Report
### fe01__family

_Description pending._

<details>
<summary>Comparison details</summary>

#### Comparison vs baseline__raw

| reference_group   | compare_group   | model_name    |   test_accuracy_mean_reference |   test_accuracy_mean_compare |   test_accuracy_mean_delta |   test_f1_mean_reference |   test_f1_mean_compare |   test_f1_mean_delta |
|:------------------|:----------------|:--------------|-------------------------------:|-----------------------------:|---------------------------:|-------------------------:|-----------------------:|---------------------:|
| baseline__raw     | fe01__family    | logreg        |                          0.786 |                        0.795 |                      0.009 |                    0.713 |                  0.721 |                0.008 |
| baseline__raw     | fe01__family    | knn           |                          0.809 |                        0.805 |                     -0.004 |                    0.742